# EVALUASI PRECISION@K — Notebook Terpadu

Notebook ini menggabungkan seluruh tahapan evaluasi dalam satu alur:

1. **Fix Ground Truth IDs** — memetakan ulang ID artikel lama ke ID baru berdasarkan pencocokan judul
2. **Load TF-IDF & Metadata** — memuat matriks TF-IDF, vektor, dan metadata artikel dari Supabase
3. **Search Semua Query** — menjalankan pencarian untuk 12 query uji bilingual
4. **Gabung dengan Ground Truth** — mencocokkan hasil pencarian dengan label final dari 3 evaluator
5. **Hitung Precision@K** — menghitung P@5, P@10, dan P@20
6. **Simpan Hasil** — menyimpan ke CSV dan Supabase
7. **Cetak Ringkasan** — menampilkan tabel hasil


## 1. Installasi Dependensi

In [1]:
%pip install supabase python-dotenv pandas numpy scipy scikit-learn PySastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Library dan Koneksi Supabase

In [2]:
# =========================================================
# CELL 2 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import json
import sys
import csv
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.metrics.pairwise import cosine_similarity

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY") or os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
EVAL_TABLE = "evaluation_precision_at_k"

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
tfidf_dir = os.path.join(base_dir, "data", "tfidf")
eval_dir = os.path.join(base_dir, "data", "evaluation")

os.makedirs(eval_dir, exist_ok=True)

print("✅ Import dan koneksi Supabase berhasil")

✅ Import dan koneksi Supabase berhasil


## 3. Load File VSM (TF-IDF Matrix)

In [3]:
# =========================================================
# CELL 3 - LOAD FILE VSM HASIL TF-IDF
# =========================================================
print("[1] Loading TF-IDF...")
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, "tfidf_matrix.npz"))

with open(os.path.join(tfidf_dir, "tfidf_terms.json"), "r", encoding="utf-8") as f:
    terms = json.load(f)

with open(os.path.join(tfidf_dir, "tfidf_doc_ids.json"), "r", encoding="utf-8") as f:
    doc_ids = json.load(f)

with open(os.path.join(tfidf_dir, "idf_scores.json"), "r", encoding="utf-8") as f:
    idf_scores = json.load(f)

tfidf_documents = pd.read_csv(os.path.join(tfidf_dir, "tfidf_documents.csv"))

doc_ids = [int(doc_id) for doc_id in doc_ids]

print("✅ File VSM berhasil dimuat")
print(f"Matrix TF-IDF: {tfidf_matrix.shape}")
print(f"Jumlah terms: {len(terms)}")
print(f"Jumlah doc_ids: {len(doc_ids)}")

[1] Loading TF-IDF...
✅ File VSM berhasil dimuat
Matrix TF-IDF: (219, 1848)
Jumlah terms: 1848
Jumlah doc_ids: 219


## 4. Load Metadata Artikel dari Supabase

In [4]:
# =========================================================
# CELL 4 - LOAD METADATA ARTIKEL DARI SUPABASE
# =========================================================
print("[2] Loading metadata from Supabase...")
all_data = []
batch_size = 1000
offset = 0

selected_columns = "id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status"

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )
    batch = response.data or []
    if not batch:
        break
    all_data.extend(batch)
    if len(batch) < batch_size:
        break
    offset += batch_size

metadata_df = pd.DataFrame(all_data)

if metadata_df.empty:
    raise ValueError("❌ Data cleaned_papers_results kosong.")

metadata_df["id"] = metadata_df["id"].astype("int64")

doc_index = pd.DataFrame({"id": doc_ids})
doc_index = doc_index.merge(metadata_df, on="id", how="left")
doc_index = doc_index.merge(
    tfidf_documents[["id", "document_text"]],
    on="id",
    how="left"
)

for col in ["title", "abstract", "authors", "source", "category", "pdf_url", "url", "scrape_status", "document_text"]:
    if col in doc_index.columns:
        doc_index[col] = doc_index[col].fillna("")

if tfidf_matrix.shape[0] != len(doc_index):
    raise ValueError("❌ Jumlah matrix TF-IDF tidak sama dengan jumlah dokumen.")

print("✅ Metadata artikel siap")
print(f"Jumlah dokumen: {len(doc_index)}")
doc_index.head(3)

[2] Loading metadata from Supabase...


✅ Metadata artikel siap
Jumlah dokumen: 219


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,document_text
0,1,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,,https://onlinelibrary.wiley.com/doi/abs/10.100...,metadata_only,what machine learning one can employ machine l...
1,2,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://www.nature.com/articles/s41579-023-009...,https://www.nature.com/articles/s41579-023-009...,metadata_only,machine learning microbiologists how evaluate ...
2,3,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning,https://ojs.aaai.org/index.php/AAAI/article/do...,https://ojs.aaai.org/index.php/AAAI/article/vi...,pdf_downloaded,amnesiac machine learning gives eu residents a...


## 5. Load Stopwords dan Stemmer

In [5]:
# =========================================================
# CELL 5 - STOPWORDS + STEMMER
# =========================================================
print("[3] Loading stopwords + stemmer...")
stop_words = get_stopwords()
stemmer = StemmerFactory().create_stemmer()
print("✅ Stopwords dan stemmer siap")

[3] Loading stopwords + stemmer...
✅ Stopwords dan stemmer siap


## 6. Definisi Fungsi Pencarian (Query → Vektor → Cosine Similarity)

Fungsi-fungsi ini merupakan inti dari mesin pencarian:
- `preprocess_query()`: membersihkan, tokenisasi, stopword removal, stemming
- `build_query_vector()`: membentuk vektor TF-IDF query
- `count_occurrence()`: menghitung frekuensi term query dalam dokumen
- `interpret_similarity()`: menginterpretasi skor cosine similarity
- `search()`: pipeline pencarian lengkap


In [6]:
# =========================================================
# CELL 6 - FUNGSI QUERY, COSINE, DAN SEARCH
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_query(query):
    cleaned = clean_text(query)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def build_query_vector(query_tokens):
    term_to_index = {term: index for index, term in enumerate(terms)}
    query_vector = np.zeros(len(terms), dtype=float)
    if not query_tokens:
        return query_vector.reshape(1, -1)
    total_terms = len(query_tokens)
    term_counts = Counter(query_tokens)
    for term, count in term_counts.items():
        if term in term_to_index:
            tf = count / total_terms
            idf = float(idf_scores.get(term, 0))
            query_vector[term_to_index[term]] = tf * idf
    return query_vector.reshape(1, -1)

def count_occurrence(document_text, query_tokens):
    document_tokens = str(document_text).split()
    return sum(document_tokens.count(term) for term in query_tokens)

def interpret_similarity(score, threshold=0.3):
    if score >= 0.7:
        return "Relevan Tinggi"
    if score >= 0.4:
        return "Relevan Sedang"
    return "Rendah"

def search(query, top_k=20, min_occurrence=0, threshold=0.3):
    query_tokens = preprocess_query(query)
    if not query_tokens:
        return pd.DataFrame()
    query_vector = build_query_vector(query_tokens)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    results = doc_index.copy()
    results["similarity_score"] = scores
    results["occurrence"] = results["document_text"].apply(
        lambda text: count_occurrence(text, query_tokens)
    )
    results = results[results["similarity_score"] > 0].copy()
    if min_occurrence > 0:
        results = results[results["occurrence"] >= min_occurrence].copy()
    results = results.sort_values("similarity_score", ascending=False)
    results = results.head(top_k).reset_index(drop=True)
    results["rank"] = range(1, len(results) + 1)
    results["threshold_relevan"] = (results["similarity_score"] > threshold).astype(int)
    results["interpretation"] = results["similarity_score"].apply(
        lambda score: interpret_similarity(score, threshold)
    )
    return results

print("✅ Fungsi search evaluasi siap")

✅ Fungsi search evaluasi siap


## 7. Query Uji Evaluasi (Bilingual)

12 query uji yang mewakili 4 kategori dengan variasi Bahasa Indonesia dan Inggris.
Nilai K = 5, 10, 20 sesuai proposal.


In [7]:
# =========================================================
# CELL 7 - QUERY UJI EVALUASI
# K = 5, 10, 20 sesuai proposal
# Query bilingual karena dataset berisi artikel Indonesia dan Inggris.
# =========================================================
queries_eval = [
    {"query": "machine learning", "kategori": "Machine Learning"},
    {"query": "pembelajaran mesin", "kategori": "Machine Learning"},
    {"query": "data mining", "kategori": "Machine Learning"},

    {"query": "web application", "kategori": "Web Application"},
    {"query": "aplikasi web", "kategori": "Web Application"},
    {"query": "sistem informasi berbasis web", "kategori": "Web Application"},

    {"query": "cyber security", "kategori": "Cyber Security"},
    {"query": "keamanan siber", "kategori": "Cyber Security"},
    {"query": "keamanan jaringan", "kategori": "Cyber Security"},

    {"query": "mobile application", "kategori": "Mobile Application"},
    {"query": "aplikasi mobile", "kategori": "Mobile Application"},
    {"query": "aplikasi android", "kategori": "Mobile Application"},
]

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)
THRESHOLD = 0.3

print(f"✅ {len(queries_eval)} query evaluasi bilingual siap")
print(f"K_VALUES = {K_VALUES}, MAX_K = {MAX_K}, THRESHOLD = {THRESHOLD}")

✅ 12 query evaluasi bilingual siap
K_VALUES = [5, 10, 20], MAX_K = 20, THRESHOLD = 0.3


## 8. Jalankan Search untuk Semua Query

In [8]:
# =========================================================
# CELL 8 - JALANKAN SEARCH UNTUK SEMUA QUERY
# =========================================================
print("[5] Running search for all queries...")
all_results = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    query = item["query"]
    kategori = item["kategori"]
    results = search(query=query, top_k=MAX_K, min_occurrence=0, threshold=THRESHOLD)
    all_results[query] = results
    print(f"  {query}: {len(results)} hasil")

    for _, row in results.iterrows():
        kemunculan_all[row["title"]] += 1
        kemunculan_kat[kategori][row["title"]] += 1

print("✅ Semua query selesai diproses")

[5] Running search for all queries...
  machine learning: 20 hasil
  pembelajaran mesin: 6 hasil
  data mining: 20 hasil
  web application: 20 hasil
  aplikasi web: 20 hasil


  sistem informasi berbasis web: 20 hasil


  cyber security: 20 hasil
  keamanan siber: 20 hasil
  keamanan jaringan: 20 hasil


  mobile application: 20 hasil


  aplikasi mobile: 20 hasil
  aplikasi android: 20 hasil
✅ Semua query selesai diproses


## 9. Fix Ground Truth IDs

Memetakan ulang ID artikel lama → ID baru dengan mencocokkan judul artikel 
antara ground truth CSV dan hasil pencarian saat ini.




In [9]:
# =========================================================
# CELL 9 - FIX GROUND TRUTH IDs
# Seluruh 225 baris penilaian evaluator digunakan (tidak difilter)
# =========================================================
print("[6] Loading ground truth CSV & Mapping IDs...")

def normalize_title(title):
    return re.sub(r'\s+', ' ', str(title).lower().strip())

gt_raw_path = os.path.join(eval_dir, "ground_truth_raw_from_sheets.csv")

gt_df = pd.read_csv(gt_raw_path)
print(f"  Ground truth raw entries: {len(gt_df)}")

# Mapping ID lama -> ID baru berdasarkan judul terhadap hasil pencarian saat ini
title_mapping = {}
for item in queries_eval:
    query = item["query"]
    current_results = all_results[query]
    current_by_title = {}
    for _, row in current_results.iterrows():
        norm = normalize_title(row["title"])
        current_by_title[norm] = int(row["id"])

    query_gt = gt_df[gt_df["Query"].str.lower().str.strip() == query.lower()]
    for _, gt_row in query_gt.iterrows():
        old_id = int(gt_row["Article ID"])
        gt_title_norm = normalize_title(gt_row.get("Judul Artikel", ""))
        if gt_title_norm in current_by_title:
            title_mapping[old_id] = current_by_title[gt_title_norm]

print(f"  IDs terpetakan: {len(title_mapping)} / {len(gt_df)}")

gt_df["Article ID (Old)"] = gt_df["Article ID"].copy()
gt_df["Article ID"] = gt_df["Article ID"].apply(lambda x: title_mapping.get(int(x), int(x)))
gt_df["ID Match"] = gt_df.apply(
    lambda r: "Matched" if int(r["Article ID (Old)"]) in title_mapping else "Unmatched", axis=1
)

# Simpan SEMUA baris (tidak difilter) agar seluruh 225 penilaian evaluator terpakai
gt_df.to_csv(os.path.join(eval_dir, "ground_truth_mapped_debug.csv"), index=False)

save_cols = [c for c in gt_df.columns if c not in ["Article ID (Old)", "ID Match"]]
gt_df[save_cols].to_csv(
    os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv"), index=False
)
print("  Saved: ground_truth_evaluation_3_evaluator.csv (seluruh baris)")

# Load GT final
gt_path = os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv")
evaluator_df = pd.read_csv(gt_path)
print(f"  Final ground truth entries: {len(evaluator_df)}")
print("✅ Ground truth siap")


[6] Loading ground truth CSV & Mapping IDs...
  Ground truth raw entries: 225
  IDs terpetakan: 118 / 225


  Saved: ground_truth_evaluation_3_evaluator.csv (seluruh baris)
  Final ground truth entries: 225
✅ Ground truth siap


## 10. Fleiss Kappa - Inter-Rater Agreement

Menghitung tingkat kesepakatan antar 3 evaluator menggunakan Fleiss Kappa.
Nilai k > 0 menunjukkan agreement di luar faktor kebetulan.


In [10]:
# =========================================================
# CELL 10 - FLEISS KAPPA
# Mengukur agreement antar 3 evaluator (Landis & Koch, 1977)
# Rumus: k = (P_bar - P_e) / (1 - P_e)
# =========================================================
import numpy as np

def fleiss_kappa_score(ratings):
    n_subjects, n_raters = ratings.shape
    n_categories = 2
    category_counts = np.zeros((n_subjects, n_categories))
    for i in range(n_subjects):
        for j in range(n_categories):
            category_counts[i, j] = float(np.sum(ratings[i] == j))
    P_i = np.sum(category_counts ** 2, axis=1) - n_raters
    P_i = P_i / (n_raters * (n_raters - 1))
    P_bar = float(np.mean(P_i))
    p_j = np.sum(category_counts, axis=0) / (n_subjects * n_raters)
    P_e = float(np.sum(p_j ** 2))
    if P_e >= 1:
        return 1.0
    return (P_bar - P_e) / (1 - P_e)

print("[10] Calculating Fleiss Kappa...")

eval_cols = ["Evaluator 1", "Evaluator 2", "Evaluator 3"]

if all(col in evaluator_df.columns for col in eval_cols):
    # Baris tanpa salah satu penilaian evaluator tidak dapat dihitung agreement-nya
    kappa_input = evaluator_df[eval_cols].dropna()
    print(f"  Baris untuk kappa (tanpa nilai kosong): {len(kappa_input)} / {len(evaluator_df)}")
    ratings = kappa_input.values
    kappa_overall = fleiss_kappa_score(ratings)
    query_to_kategori = {item["query"]: item["kategori"] for item in queries_eval}
    evaluator_df["kategori"] = evaluator_df["Query"].str.lower().str.strip().map(query_to_kategori)
    print("=" * 60)
    print("FLEISS KAPPA - Inter-Rater Agreement")
    print("=" * 60)
    print(f"Keseluruhan: k = {kappa_overall:.3f}")
    print()
    print("Per Kategori:")
    for kategori in ["Machine Learning", "Web Application", "Cyber Security", "Mobile Application"]:
        mask = evaluator_df["kategori"] == kategori
        if mask.sum() > 0:
            kat_ratings = evaluator_df.loc[mask, eval_cols].dropna().values
            if len(kat_ratings) > 0:
                kat_kappa = fleiss_kappa_score(kat_ratings)
                print(f"  {kategori}: k = {kat_kappa:.3f} (n={len(kat_ratings)})")
    print()
    print("Interpretasi (Landis and Koch, 1977):")
    print("  <0 = poor, 0.00-0.20 = slight, 0.21-0.40 = fair, 0.41-0.60 = moderate,")
    print("  0.61-0.80 = substantial, 0.81-1.00 = almost perfect")
else:
    print("Kolom evaluator tidak ditemukan di ground truth.")
    print(f"Kolom tersedia: {list(evaluator_df.columns)}")

print("Fleiss Kappa selesai")


[10] Calculating Fleiss Kappa...
  Baris untuk kappa (tanpa nilai kosong): 224 / 225
FLEISS KAPPA - Inter-Rater Agreement
Keseluruhan: k = 0.176

Per Kategori:
  Machine Learning: k = 0.385 (n=45)
  Web Application: k = 0.147 (n=60)
  Cyber Security: k = -0.007 (n=60)
  Mobile Application: k = 0.062 (n=59)

Interpretasi (Landis and Koch, 1977):
  <0 = poor, 0.00-0.20 = slight, 0.21-0.40 = fair, 0.41-0.60 = moderate,
  0.61-0.80 = substantial, 0.81-1.00 = almost perfect
Fleiss Kappa selesai


## 12. Susun Hasil Pencarian dan Gabung dengan Ground Truth

In [11]:
# =========================================================
# CELL 12 - SUSUN HASIL PENCARIAN + MERGE DENGAN GROUND TRUTH
# =========================================================
print("[7] Building search results dataframe...")
rows = []

#ambil hasil search 12 query digabung jadi satu dataframe
for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query]
    for _, row in results.iterrows():
        rows.append({
            "query": query,
            "kategori_query": kategori_query,
            "rank": int(row["rank"]),
            "article_id": int(row["id"]),
            "article_category": row["category"],
            "title": row["title"],
            "abstract": row["abstract"],
            "similarity_score": float(row["similarity_score"]),
            "threshold_relevan": int(row["threshold_relevan"]),
        })

hasil_pencarian_eval_df = pd.DataFrame(rows)
print(f"  Total search results: {len(hasil_pencarian_eval_df)}")

print("[8] Merging with ground truth...")
evaluator_df["query_key"] = evaluator_df["Query"].astype(str).str.lower().str.strip()
evaluator_df["article_id"] = pd.to_numeric(evaluator_df["Article ID"], errors="coerce").astype("Int64")

#merge hasil pencarian dengan ground truth evaluator
gt_df = hasil_pencarian_eval_df.copy()
gt_df["query_key"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = pd.to_numeric(gt_df["article_id"], errors="coerce").astype("Int64")

eval_cols = ["query_key", "article_id", "Label Final", "Status Final",
            "Evaluator 1", "Evaluator 2", "Evaluator 3", "Catatan Evaluator",
            "Threshold Awal", "Similarity Score"]
merge_cols = [c for c in eval_cols if c in evaluator_df.columns]
gt_df = gt_df.merge(evaluator_df[merge_cols], on=["query_key", "article_id"], how="left")

#ambil kolom label final untuk dijadikan kolom relevan (1 = relevan, 0 = tidak relevan)
gt_df["relevan"] = pd.to_numeric(gt_df["Label Final"], errors="coerce").fillna(0).astype(int)

#rapiing kolom untuk memastikan tipe data konsisten
gt_df["query"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = gt_df["article_id"].astype("int64")
gt_df["relevan"] = gt_df["relevan"].astype(int)

print(f"  Total after merge: {len(gt_df)}")
print(f"  Distribusi relevan:\n{gt_df['relevan'].value_counts().sort_index()}")
print("✅ Data siap untuk perhitungan Precision@K")

[7] Building search results dataframe...
  Total search results: 226
[8] Merging with ground truth...


  Total after merge: 227
  Distribusi relevan:
relevan
0     55
1    172
Name: count, dtype: int64
✅ Data siap untuk perhitungan Precision@K


## 13. Perhitungan Precision@K

Untuk setiap query, hitung Precision@5, @10, dan @20 dengan rumus:

$$P@K = \frac{\text{jumlah artikel relevan di peringkat 1..K}}{K}$$

In [12]:
# =========================================================
# CELL 13 - HITUNG PRECISION@K
# Dihitung langsung dari 225 penilaian evaluator (rank hasil pencarian
# sistem vs label relevansi manusia), bukan dari threshold similarity.
# =========================================================
print("[9] Calculating Precision@K (human judgment)...")

eval_rows = []
for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    q_gt = evaluator_df[
        evaluator_df["Query"].astype(str).str.lower().str.strip() == query.lower()
    ].sort_values("Rank").copy()
    q_gt["relevan"] = pd.to_numeric(q_gt["Label Final"], errors="coerce").fillna(0).astype(int)
    row_eval = {"Query": query, "Kategori": kategori_query, "Retrieved": int(len(q_gt))}
    for k in K_VALUES:
        top_k = q_gt.head(k)
        relevant_k = int(top_k["relevan"].sum())
        row_eval[f"Relevan@{k}"] = relevant_k
        row_eval[f"P@{k}"] = round(relevant_k / k, 4)
    eval_rows.append(row_eval)

eval_df = pd.DataFrame(eval_rows)
eval_path = os.path.join(eval_dir, "tabel2_precision_at_k.csv")
eval_df.to_csv(eval_path, index=False)
print(f"  Tabel 2 saved: {eval_path}")

print("\n  Precision@K per query:")
print(eval_df.to_string(index=False))


[9] Calculating Precision@K (human judgment)...
  Tabel 2 saved: D:\Tugas Akhir\paperci_artikel\backend\data\evaluation\tabel2_precision_at_k.csv

  Precision@K per query:
                        Query           Kategori  Retrieved  Relevan@5  P@5  Relevan@10  P@10  Relevan@20  P@20
             machine learning   Machine Learning         20          5  1.0          10   1.0          20  1.00
           pembelajaran mesin   Machine Learning          5          4  0.8           4   0.4           4  0.20
                  data mining   Machine Learning         20          3  0.6           5   0.5          10  0.50
              web application    Web Application         20          5  1.0          10   1.0          19  0.95
                 aplikasi web    Web Application         20          5  1.0          10   1.0          18  0.90
sistem informasi berbasis web    Web Application         20          4  0.8           8   0.8          15  0.75
               cyber security     Cyber Secu

## 14. Rata-rata Precision@K

Hitung rata-rata P@K untuk semua query (keseluruhan dan per kategori).


In [ ]:
# =========================================================
#  - RATA-RATA PRECISION@K
# =========================================================
print("[10] Calculating averages...")

# --- Rata-rata Keseluruhan ---
summary_rows = []
for k in K_VALUES:
    mean_val = round(eval_df[f"P@{k}"].mean(), 4)
    summary_rows.append({
        "Metrik": f"Mean P@{k}",
        "Nilai": mean_val,
        "Persentase": round(mean_val * 100, 2)
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(eval_dir, "rata_rata_keseluruhan.csv")
summary_df.to_csv(summary_path, index=False)

print("\n  Rata-rata keseluruhan:")
print(summary_df.to_string(index=False))

# --- Rata-rata per Kategori ---
kat_avg_rows = []
for kategori, group in eval_df.groupby("Kategori"):
    row = {"Kategori": kategori}
    for k in K_VALUES:
        row[f"Rata-rata P@{k}"] = round(group[f"P@{k}"].mean(), 4)
    kat_avg_rows.append(row)

kat_avg_df = pd.DataFrame(kat_avg_rows)
kat_avg_path = os.path.join(eval_dir, "rata_rata_per_kategori.csv")
kat_avg_df.to_csv(kat_avg_path, index=False)

print("\n  Rata-rata per kategori:")
print(kat_avg_df.to_string(index=False))

[10] Calculating averages...

  Rata-rata keseluruhan:
   Metrik  Nilai  Persentase
 Mean P@5 0.9000       90.00
Mean P@10 0.8667       86.67
Mean P@20 0.8167       81.67

  Rata-rata per kategori:
          Kategori  Rata-rata P@5  Rata-rata P@10  Rata-rata P@20
    Cyber Security         0.9333          0.9667          0.9833
  Machine Learning         0.8000          0.6333          0.5667
Mobile Application         0.9333          0.9333          0.8500
   Web Application         0.9333          0.9333          0.8667


## 15. Simpan Hasil Evaluasi ke Supabase

Menyimpan nilai Precision@K ke tabel `evaluation_precision_at_k` di Supabase.


In [14]:
# =========================================================
# CELL 13 - SIMPAN KE SUPABASE
# =========================================================
print("[11] Saving to Supabase...")
ts = datetime.now(timezone.utc).isoformat()
rows_db = []
for _, row in eval_df.iterrows():
    query = row["Query"]
    for k in K_VALUES:
        rows_db.append({
            "compared_text": query,
            "k": int(k),
            "retrieved_count": int(row["Retrieved"]),
            "relevant_retrieved": int(row[f"Relevan@{k}"]),
            "precision_at_k": float(row[f"P@{k}"]),
            "updated_at": ts
        })

for start in range(0, len(rows_db), 500):
    batch = rows_db[start:start + 500]
    supabase.table(EVAL_TABLE).upsert(batch, on_conflict="compared_text,k").execute()

print(f"  Saved {len(rows_db)} rows to {EVAL_TABLE}")
print("\n✅ Evaluation complete!")

[11] Saving to Supabase...


  Saved 36 rows to evaluation_precision_at_k

✅ Evaluation complete!


## 16. Ringkasan Hasil

Menampilkan contoh hasil query "cyber security" Top-5 dan seluruh tabel Precision@K.


In [15]:
# =========================================================
# CELL 14 - EKSPERIMEN PENENTUAN THRESHOLD COSINE SIMILARITY
# Membandingkan prediksi otomatis (Similarity Score > threshold)
# terhadap Label Final (human judgment) dari 3 evaluator.
# =========================================================
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

print("[12] Threshold experiment...")

thr_df = evaluator_df.dropna(subset=["Similarity Score", "Label Final"]).copy()
thr_df["y_true"] = thr_df["Label Final"].astype(int)
print(f"  Data eksperimen: {len(thr_df)} baris")
print(f"  Base rate: relevan={int(thr_df['y_true'].sum())} ({thr_df['y_true'].mean()*100:.1f}%), tidak relevan={int((thr_df['y_true']==0).sum())}")
print()

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
thr_rows = []
print("  Threshold | Accuracy | Precision | Recall | F1-Score | TP | FP | FN | TN")
print("  --------- | -------- | --------- | ------ | -------- | -- | -- | -- | --")
for t in thresholds:
    pred = (thr_df["Similarity Score"] > t).astype(int)
    cm = confusion_matrix(thr_df["y_true"], pred, labels=[1, 0])
    tp, fn, fp, tn = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    acc = accuracy_score(thr_df["y_true"], pred)
    prec = precision_score(thr_df["y_true"], pred, zero_division=0)
    rec = recall_score(thr_df["y_true"], pred, zero_division=0)
    f1 = f1_score(thr_df["y_true"], pred, zero_division=0)
    thr_rows.append({
        "Threshold": t, "Accuracy": round(acc, 4), "Precision": round(prec, 4),
        "Recall": round(rec, 4), "F1-Score": round(f1, 4),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })
    print(f"  {t:.2f}      | {acc:.4f}  | {prec:.4f}   | {rec:.4f}  | {f1:.4f}     | {tp:2d} | {fp:2d} | {fn:3d} | {tn:2d}")

thr_exp_df = pd.DataFrame(thr_rows)
thr_path = os.path.join(eval_dir, "threshold_experiment.csv")
thr_exp_df.to_csv(thr_path, index=False)
print(f"\n  Saved: {thr_path}")
print("✅ Eksperimen threshold selesai")

print("\n" + "=" * 70)
print("EKSPERIMEN THRESHOLD COSINE SIMILARITY")
print("=" * 70)
print("Perbandingan prediksi otomatis (Similarity Score > threshold) vs Label Final (human judgment):")
print(thr_exp_df.to_string(index=False))
print("\nInterpretasi:")
print("  - Threshold 0.30 menghasilkan Accuracy, Precision, Recall, dan F1-score terbaik.")
print("  - Berdasarkan hasil pengujian menggunakan data pembanding (Ground Truth), threshold 0.30 dipilih sebagai threshold yang digunakan pada sistem.")
print("\n✅ Semua hasil siap untuk laporan.")


[12] Threshold experiment...
  Data eksperimen: 225 baris
  Base rate: relevan=196 (87.1%), tidak relevan=29

  Threshold | Accuracy | Precision | Recall | F1-Score | TP | FP | FN | TN
  --------- | -------- | --------- | ------ | -------- | -- | -- | -- | --
  0.30      | 0.5200  | 1.0000   | 0.4490  | 0.6197     | 88 |  0 | 108 | 29
  0.40      | 0.3067  | 1.0000   | 0.2041  | 0.3390     | 40 |  0 | 156 | 29
  0.50      | 0.2133  | 1.0000   | 0.0969  | 0.1767     | 19 |  0 | 177 | 29
  0.60      | 0.1467  | 1.0000   | 0.0204  | 0.0400     |  4 |  0 | 192 | 29
  0.70      | 0.1333  | 1.0000   | 0.0051  | 0.0102     |  1 |  0 | 195 | 29

  Saved: D:\Tugas Akhir\paperci_artikel\backend\data\evaluation\threshold_experiment.csv
✅ Eksperimen threshold selesai

EKSPERIMEN THRESHOLD COSINE SIMILARITY
Perbandingan prediksi otomatis (Similarity Score > threshold) vs Label Final (human judgment):
 Threshold  Accuracy  Precision  Recall  F1-Score  TP  FP  FN  TN
       0.3    0.5200        1.0  

In [16]:
# =========================================================
# CELL 15 - CETAK RINGKASAN HASIL
# =========================================================
print("=" * 70)
print("CONTOH: Hasil Pencarian Query 'cyber security' pada Top-5")
print("=" * 70)

contoh = evaluator_df[
    evaluator_df["Query"].astype(str).str.lower().str.strip() == "cyber security"
].sort_values("Rank").head(5)
for _, row in contoh.iterrows():
    print(f"Rank {int(row['Rank'])} | ID={int(row['Article ID'])}")
    print(f"  Judul: {str(row['Judul Artikel'])[:80]}...")
    thr = 'Ya' if int(row['Threshold Awal']) == 1 else 'Tidak'
    print(f"  Similarity: {row['Similarity Score']:.4f} | Threshold Awal: {thr} | Human Judgment: {row['Status Final']}")
    print()

print("=" * 70)
print("TABEL 2: Precision@K per Query")
print("=" * 70)
print(eval_df.to_string(index=False))

print("\n" + "=" * 70)
print("RATA-RATA PRECISION@K")
print("=" * 70)
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("RATA-RATA PER KATEGORI")
print("=" * 70)
print(kat_avg_df.to_string(index=False))

print("\n✅ Semua hasil siap untuk laporan.")


CONTOH: Hasil Pencarian Query 'cyber security' pada Top-5
Rank 1 | ID=155
  Judul: A systematic literature review on the cyber security...
  Similarity: 0.8084 | Threshold Awal: Ya | Human Judgment: Relevan

Rank 2 | ID=162
  Judul: Analysis of cyber security knowledge gaps based on cyber security body of knowle...
  Similarity: 0.6292 | Threshold Awal: Ya | Human Judgment: Relevan

Rank 3 | ID=167
  Judul: Artificial intelligence-based cyber security in the context of industry 4.0—a su...
  Similarity: 0.5492 | Threshold Awal: Ya | Human Judgment: Relevan

Rank 4 | ID=174
  Judul: Review on cyber-physical and cyber-security system in smart grid: Standards, pro...
  Similarity: 0.5219 | Threshold Awal: Ya | Human Judgment: Relevan

Rank 5 | ID=153
  Judul: A comprehensive review study of cyber-attacks and cyber security; Emerging trend...
  Similarity: 0.5198 | Threshold Awal: Ya | Human Judgment: Relevan

TABEL 2: Precision@K per Query
                        Query           Kategori 